<img src="https://hilpisch.com/tpq_logo_bic.png" width="20%" align="right">

# Python & AI for Rates, Bonds, and Credit

## Chapter 17 · Reduced-Form Credit Models and CDS

[Open in Google Colab](https://colab.research.google.com/github/yhilpisch/ratescode/blob/main/notebooks/ch17_reduced_form_credit_models_cds.ipynb)

&copy; Dr. Yves J. Hilpisch<br>
AI-Powered by different LLMs<br>
The Python Quants GmbH | https://tpq.io<br>
https://hilpisch.com | https://linktr.ee/dyjh

## Learning goals

This notebook turns the reduced-form machinery of Chapter 17 into running code. You will

- convert a constant hazard rate into survival and period default probabilities,
- build the premium and protection legs of a credit default swap from discount factors, accrual periods, and the survival curve,
- solve for the par CDS spread that balances the two legs,
- compare the exact par spread with the shortcut spread given by hazard times loss given default, and
- read repricing residuals as calibration diagnostics rather than as errors to hide.

All default quantities here are risk-neutral: implied by spreads and recovery, not observed as historical default frequencies.

## Prepare the companion repository

This setup cell works with a local checkout or a fresh Google Colab runtime. In Colab it clones the public companion repository, then sets `ROOT` so the chapter code and frozen data snapshots are found without manual path edits.

In [ ]:
from pathlib import Path
import importlib.util
import subprocess
import sys

REPO_URL = "https://github.com/yhilpisch/ratescode.git"
candidates = [Path.cwd(), Path.cwd().parent,
              Path("/content/07_ratescode")]
ROOT = next((p.resolve() for p in candidates
             if (p / "code").is_dir()
             and (p / "data").is_dir()), None)
if ROOT is None:
    ROOT = Path("/content/07_ratescode")
    subprocess.run(["git", "clone", "--depth", "1",
                    REPO_URL, str(ROOT)], check=True)

packages = {"numpy": "numpy", "pandas": "pandas",
            "matplotlib": "matplotlib"}
missing = [package for module, package in packages.items()
           if importlib.util.find_spec(module) is None]
if missing:
    subprocess.run([sys.executable, "-m", "pip", "install",
                    *missing], check=True)
print(f"Repository: {ROOT}")
print(f"Data files: {len(list((ROOT / 'data').glob('*.csv')))}")


## Run the chapter example

The cell below runs the Chapter 17 script. It prints the first rows of the quarterly survival curve for a constant 2.5% hazard rate, then reprices the frozen one-year and two-year CDS quotes using the shortcut hazard, spread over one minus recovery, applied independently at each maturity. The nonzero residuals show how much of each quote a flat-hazard shortcut leaves unexplained.

In [ ]:
SCRIPT = ROOT / "code" / "ch17_reduced_form_credit_models_cds.py"
print(SCRIPT.relative_to(ROOT))

result = subprocess.run(
    [sys.executable, str(SCRIPT)],
    cwd=ROOT,
    capture_output=True,
    text=True,
    check=True,
)
print(result.stdout)

## Interpretation checklist

- Verify the survival column: `exp(-0.025 * t)` falls from 0.993769 at 0.25 years to 0.975310 at 1.00 year, and the default_prob column is the period-by-period decrement.
- Check the premium-leg structure: spread times accrual times discount factor times survival, summed over premium dates.
- Check that the protection leg uses loss given default, 0.60 at 40% recovery, times discounted period default probabilities, not recovery itself.
- Read the residuals, -0.000029 and -0.000112 per unit notional, as the size of the shortcut's error at each maturity; premiums also omit accrued interest on default, so even a fitted curve keeps a timing-convention residual.
- State that the reported hazard rates, 0.0200 and 0.028333, are risk-neutral intensities, not historical default frequencies.

In [ ]:
import pandas as pd

DATA = ROOT / "data"
chapter_files = sorted(DATA.glob("ch17_*.csv"))
summary = pd.DataFrame([
    {"file": path.name,
     "rows": len(frame := pd.read_csv(path)),
     "columns": len(frame.columns),
     "missing_cells": int(frame.isna().sum().sum()),
     "fields": str(list(frame.columns))}
    for path in chapter_files
])
summary


## Chapter-specific checks

Translate each CDS quote into an independent flat-hazard approximation and its corresponding survival probability. These rows are not a jointly calibrated survival curve.

In [ ]:
import numpy as np

quotes = pd.read_csv(DATA / "ch17_cds_quotes.csv")
recovery = pd.read_csv(DATA / "ch17_recovery_assumptions.csv").recovery.iloc[0]
quotes["hazard_approx"] = quotes.spread_bp / 10000 / (1 - recovery)
quotes["survival"] = np.exp(-quotes.hazard_approx * quotes.maturity)
quotes.round(6)

## Mini-lab: The Par CDS Spread From a Survival Curve

A one-year CDS pays quarterly premiums while the reference name survives and compensates the protection buyer with loss given default on default. With a constant hazard rate of 2.5% per year and recovery of 40% of face, one survival curve drives both legs: the premium annuity is the sum of accrual times discount factor times survival, and the protection leg is 60% of the discounted period default probabilities. The par spread is the ratio of the two legs, and the shortcut spread given by hazard times LGD gives a quick comparison.

In [ ]:
import numpy as np

hazard, recovery = 0.025, 0.40  # intensity per year, recovery of face
time = np.array([0.25, 0.50, 0.75, 1.00])  # premium dates in years
discount = np.array([0.990, 0.980, 0.970, 0.960])
survival = np.exp(-hazard * time)  # survival probability
default_prob = np.r_[1.0, survival[:-1]] - survival  # per period
annuity = float((0.25 * discount * survival).sum())  # premium annuity
protection = float((1 - recovery) * (discount * default_prob).sum())
print(f"protection leg: {protection:.6f}")
print(f"par CDS spread: {protection / annuity * 10_000:.1f} bp")
print(f"shortcut spread: {hazard * (1 - recovery) * 10_000:.1f} bp")

### What to observe

The protection leg is 0.014445 per unit notional against a premium annuity of 0.959984, so the par CDS spread is 150.5 bp. The shortcut hazard times LGD gives 150.0 bp, half a basis point off, because it ignores discounting and the timing of default within each period. At the par spread the two legs balance by construction.

## Exercises

1. **Survival by hand.** Compute the survival probability to 2 years for a 2.5% constant hazard. *Hint:* `exp(-0.05)`.
2. **Leg balance.** Recompute both legs at the mini-lab's par spread and confirm the premium leg equals the protection leg to six decimals.
3. **Recovery sensitivity.** Raise recovery from 40% to 50% and recompute the par spread. Explain the direction of the move. *Hint:* protection scales with one minus recovery.
4. **Hazard from a quote.** Given a 120 bp one-year quote and 40% recovery, back out the shortcut hazard and compare it with the script's 0.02.
5. **Term structure.** The frozen quotes rise from 120 bp at one year to 170 bp at two years. Explain what that slope says about the market-implied default path.
6. **Risk-neutral caveat.** In two sentences, explain why a hazard rate bootstrapped from CDS quotes can exceed the historical default frequency for the same rating bucket.

For more detail, see Chapter 17 of *Python & AI for Rates, Bonds, and Credit*.

<img src="https://hilpisch.com/tpq_logo_bic.png" width="20%" align="right">